# 1. Question 1: Missing Data Mechanisms

### Theory
- **MCAR (Missing Completely at Random):** Missingness is entirely random and unrelated to observed or unobserved data (e.g., lab sample dropped on floor).
- **MAR (Missing at Random):** Missingness depends systematically on observed features, but not on the missing value itself (e.g., men reporting income less frequently).
- **MNAR (Missing Not at Random):** Missingness depends directly on the unobserved missing value itself (e.g., high earners withholding income figures).

### Business Impact
Misdiagnosing MNAR as MCAR and applying mean imputation biases business estimates.

### Risks
Imputing MNAR data without adding missingness indicators distorts target relationships.

### Decision Rules
- If data is MNAR, use pattern indicators (`MissingIndicator`) alongside imputation.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, MissingIndicator

# Simulate MNAR feature vector
data = pd.DataFrame({"Salary": [50000, np.nan, 120000, np.nan, 85000]})

# Solution: Impute mean + capture missingness indicator flag
indicator = MissingIndicator()
missing_flags = indicator.fit_transform(data)

imputer = SimpleImputer(strategy="mean")
imputed_salary = imputer.fit_transform(data)

result = pd.DataFrame({
    "Imputed_Salary": imputed_salary.ravel(),
    "Salary_Was_Missing": missing_flags.ravel()
})
display(result)

,Imputed_Salary,Salary_Was_Missing
0,50000.0,False
1,85000.0,True
2,120000.0,False
3,85000.0,True
4,85000.0,False


# 2. Question 2: Data Leakage Risks & Prevention

### Theory
Data leakage happens when information outside the training set contaminates the model during training.

### Business Impact
Leads to artificially high offline validation scores (e.g., 99% accuracy) that collapse upon live deployment.

### Risks
Features generated post-outcome (e.g., `Cancellation_Refund_Amount`) leaking into churn prediction models.

### Decision Rules
- Drop all post-event features and enforce a strict train/test split prior to any preprocessing.

In [2]:
# Code Challenge: Detect and drop leaky target features
df_leak_example = pd.DataFrame({
    "Age": [25, 45, 35, 50],
    "Monthly_Fee": [50, 80, 65, 100],
    "Refund_Processed_Post_Cancel": [10, 0, 15, 0], # LEAK FEATURE
    "Churn": [1, 0, 1, 0]
})

# Correct Solution: Filter leaky features out of training matrix
leaky_cols = ["Refund_Processed_Post_Cancel"]
X_clean_assessment = df_leak_example.drop(columns=["Churn"] + leaky_cols)

print("Clean Features Matrix (Leaky Columns Dropped):")
display(X_clean_assessment)

Clean Features Matrix (Leaky Columns Dropped):


,Age,Monthly_Fee
0,25,50
1,45,80
2,35,65
3,50,100


# 3. Question 3: IQR Outlier Capping Implementation

### Theory
Calculates the Interquartile Range ($IQR = Q_3 - Q_1$) and caps continuous values strictly within $[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$.

### Business Impact
Prevents extreme values from distorting model training while preserving sample counts.

### Risks
Capping valid extreme entries can reduce sensitivity to rare business events.

### Decision Rules
- Apply IQR capping to right-skewed non-Gaussian continuous variables.

In [3]:
# Practical Solution Challenge: Write an IQR Outlier Capper Function
def cap_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return np.clip(series, lower_bound, upper_bound)

sample_data = pd.Series([10, 12, 14, 15, 18, 20, 500]) # 500 is extreme outlier
capped_data = cap_outliers_iqr(sample_data)

print("Original Max:", sample_data.max())
print("Capped Max (Post IQR Treatment):", capped_data.max())

Original Max: 500
Capped Max (Post IQR Treatment): 28


# 4. Question 4: End-to-End Pipeline Assembly

### Theory
Assembles an end-to-end preprocessing pipeline that imputes and scales numeric features while imputing and one-hot encoding categorical features.

### Business Impact
Provides a single, standardized pipeline for preprocessing numerical and categorical inputs.

### Risks
Missing categories or mismatched column data types throw errors if sub-pipelines are misconfigured.

### Decision Rules
- Verify pipeline transforms work cleanly on sample datasets before deploying to production models.

In [4]:
# Practical Solution Challenge: Build full pipeline from scratch
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Test Data
df_test_sample = pd.DataFrame({
    "Age": [25, np.nan, 35, 45],
    "Score": [100.0, 200.0, np.nan, 400.0],
    "City": ["NY", "SF", np.nan, "NY"]
})

# Pipeline Definition
num_sub = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_sub = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder())])

final_assessment_pipeline = ColumnTransformer([
    ("num", num_sub, ["Age", "Score"]),
    ("cat", cat_sub, ["City"])
])

processed_array = final_assessment_pipeline.fit_transform(df_test_sample)
print("Processed Assessment Matrix Shape:", processed_array.shape)
print("Notebook 17 execution completed successfully!")

Processed Assessment Matrix Shape: (4, 4)
Notebook 17 execution completed successfully!
